# PYNQ 加速器联调

按顺序执行。Python 必须运行在开发板内核。默认只检查环境并运行离线比较；硬件步骤由 `RUN_HARDWARE` 显式启用。

支持 AXI DMA simple mode、一入一出。示例为 DMA→FIFO→DMA 恒等回环；其他算法需要替换输入与参考输出。参阅 README.md 与 references/。


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = Path.cwd()
assert (ROOT / "scripts/pynq_debug.py").is_file(), "请在技能包目录打开 Notebook"
sys.path.insert(0, str(ROOT / "scripts"))
from pynq_debug import environment, overlay_metadata, compare, run_dma, save_report, TransferFailure
print(json.dumps(environment(), indent=2, ensure_ascii=False))


## 1. 参数和参考结果

bit/hwh 必须同名、同目录且来自同次构建。IP 名称从元数据确认。`RUN_HARDWARE=True` 会下载位流并替换当前 PL；仅在授权的独占测试环境启用。位流使用本板配套的DDR/MIO配置。


In [ ]:
BITFILE = ROOT / "overlays/dma_tutorial.bit"
DMA_NAME = "axi_dma_0"
RUN_HARDWARE = False
TIMEOUT_S = 3.0
ATOL = 0.0
RTOL = 0.0
ACCELERATOR = None  # 启动型 IP 填入已核对的 {"ip": ..., "writes": [[offset, value], ...]}

x = np.arange(100, dtype=np.uint32) + np.uint32(0xabcd0000)
expected = x.copy()  # 仅适用于恒等回环；自己的算法必须替换参考模型
print({"input_shape": x.shape, "input_dtype": str(x.dtype), "tx_bytes": x.nbytes,
       "output_shape": expected.shape, "rx_bytes": expected.nbytes})
print("Offline comparator:", compare(x, expected, atol=ATOL, rtol=RTOL))


## 2. 检查 Overlay 元数据

`download=False` 不下载位流。这里只读取 ip_dict，避免提前实例化 DMA 驱动。先检查 IP 类型、长度配置、simple mode 和双通道。


In [ ]:
ol = None
if BITFILE.is_file() and BITFILE.with_suffix(".hwh").is_file():
    from pynq import Overlay
    ol = Overlay(str(BITFILE), download=False)
    inventory = overlay_metadata(ol)
    print("Available IP:", list(inventory))
    assert DMA_NAME in inventory, "DMA 名称不在 HWH 中"
    print(json.dumps(inventory[DMA_NAME], indent=2, default=str))
elif RUN_HARDWARE:
    raise FileNotFoundError("缺少同名 bit/hwh 文件")
else:
    print("SKIPPED: 尚未提供 bit/hwh；离线比较仍可使用")


## 3. 下载与单次传输

接收先行，再启动加速器并发送。完成超时不覆盖驱动初始化/硬件总线挂起。异常时保存 JSON 报告；若提示缓冲区被隔离，先复位/重启板卡，不要直接重启内核或释放缓冲区。


In [ ]:
report = None
if RUN_HARDWARE:
    assert ol is not None
    # A fresh object avoids cached DMA drivers from a previous download.
    ol = Overlay(str(BITFILE), download=True)
    try:
        actual, report = run_dma(ol, DMA_NAME, x, expected, timeout_s=TIMEOUT_S,
                                 atol=ATOL, rtol=RTOL, accelerator=ACCELERATOR)
    except TransferFailure as exc:
        print("Error report:", save_report(exc.report, ROOT / "reports"))
        raise
    print(json.dumps(report, indent=2, default=str))
    print("Saved:", save_report(report, ROOT / "reports"))
    assert report["status"] == "pass", "传输完成但参考结果不一致"
else:
    print("SKIPPED: RUN_HARDWARE=False，没有下载位流或发起传输")


## 4. 改换数据验证复用性

下列测试只适用于恒等回环。包含随机 uint32 和不同包长；最大长度由 HWH 决定。其他算法同步替换参考输出和回归断言。

transfer_wall_ms 从Python发起传输计到完成等待与同步。换板后重新构建位流，并执行迁移验收。


In [ ]:
if RUN_HARDWARE:
    rng = np.random.default_rng(20261025)
    results = []
    for n in (1, 16, 100, 1024):
        sample = rng.integers(0, 2**32, size=n, dtype=np.uint32)
        try:
            _, one = run_dma(ol, DMA_NAME, sample, sample.copy(), timeout_s=TIMEOUT_S)
        except TransferFailure as exc:
            print("Error report:", save_report(exc.report, ROOT / "reports"))
            raise
        one["case"] = {"pattern": "random_uint32", "n": n, "seed": 20261025}
        results.append(one)
        print(n, one["status"], one["received_bytes"])
        save_report(one, ROOT / "reports")
        assert one["status"] == "pass"
else:
    print("SKIPPED: 硬件回归尚未启用")
